##  Setup & Configuration

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid", palette="muted")

In [3]:
DATA_DIR        = Path("Data")
END_DATE        = "2026-01-31"
DELAY_THRESHOLD = 5
WEATHER_CACHE   = DATA_DIR / "weather_toronto.csv"
EXPORT_PATH     = DATA_DIR / "bus_powerbi_1.csv"
EXPORT_MODEL    = DATA_DIR / "bus_model_export_1.csv"

## Handling & Merging Data

Notice that the raw data consists of some inconsistencies between 2022-2024 and 2025+. The differences are noted below:

| | 2022-2024 XLSX | 2025+ CSV |
|---|---|---|
| Route | `Route` — integer (mostly) | `Line` — string e.g. "102 MARKHAM ROAD" |
| Location | `Location` — intersection name | `Station` — stop name |
| Incident type | `Incident` — plain English (15 categories) | `Code` — bus-specific code e.g. "EFO", "MFSH" |
| Direction | `Direction` — N/S/E/W + junk | `Bound` — N/S/E/W + junk |

ALso, the Code Descriptions file, `Bus Code Descriptions.csv` has a Windows-1252 encoding artefact: `â` should be read as `-` (em-dash corruption).

In [4]:
df_bus_codes = pd.read_csv(DATA_DIR / "Bus Code Descriptions.csv")
# Fix encoding artefact: the 3-character sequence U+00E2 U+0080 U+0093 is a
# double-encoded en-dash (–). We must replace all three chars together in one go
# replacing just the first char leaves two invisible control chars behind.
CORRUPT = "â"
df_bus_codes["DESCRIPTION"] = (
    df_bus_codes["DESCRIPTION"]
    .str.replace(CORRUPT, " - ", regex=False)
    .str.strip()
)
CODE_TO_DESC = dict(zip(df_bus_codes["CODE"], df_bus_codes["DESCRIPTION"]))
print(f"{len(df_bus_codes)} bus delay codes")
pd.set_option("display.max_rows", None)
print(df_bus_codes[["CODE","DESCRIPTION"]].to_string(index=False))

46 bus delay codes
 CODE                                   DESCRIPTION
  EFB                                          BODY
EFCAN                                  CANCELLATION
  EFD                                         DOORS
 EFDB                                   DISC BRAKES
 EFHV                                  HIGH VOLTAGE
EFHVA                              HEAT / VENT / AC
 EFLV                                   LOW VOLTAGE
  EFO                                         OTHER
  EFP                                    PROPULSION
 EFRA                                   RAMP ISSUES
  EFT                                        TRUCKS
 EFTB                                  TRACK BRAKES
 MFCN                                      CLEANING
 MFDV                                  ON DIVERSION
MFESA              NO OPERATOR AVAILABLE DUE TO ESA
 MFFD                                  FARE DISPUTE
 MFLD                                LABOUR DISPUTE
  MFO                                        

In [6]:
# loading xslx files (2022-2024)
print("Loading 2022-2024 XLSX files...")
xlsx_sources = [
    (DATA_DIR / "ttc-bus-delay-data-2022.xlsx", "2022"),
    (DATA_DIR / "ttc-bus-delay-data-2023.xlsx", "2023"),
    (DATA_DIR / "ttc-bus-delay-data-2024.xlsx", "2024"),
]
dfs_xlsx = [pd.read_excel(p, sheet_name=s) for p, s in xlsx_sources]
df_xlsx = pd.concat(dfs_xlsx, ignore_index=True)
print(f"  2022-2024 combined : {len(df_xlsx):,} rows")
print(f"  Columns : {df_xlsx.columns.tolist()}")
print(df_xlsx.head(3).to_string())

Loading 2022-2024 XLSX files...
  2022-2024 combined : 174,557 rows
  Columns : ['Date', 'Route', 'Time', 'Day', 'Location', 'Incident', 'Min Delay', 'Min Gap', 'Direction', 'Vehicle']
        Date Route   Time       Day                Location               Incident  Min Delay  Min Gap Direction  Vehicle
0 2022-01-01   320  02:00  Saturday        YONGE AND DUNDAS          General Delay          0        0       NaN     8531
1 2022-01-01   325  02:00  Saturday  OVERLEA AND THORCLIFFE              Diversion        131      161         W     8658
2 2022-01-01   320  02:00  Saturday       YONGE AND STEELES  Operations - Operator         17       20         S        0


In [7]:
# loading 2025+ csv file
df_csv = pd.read_csv(DATA_DIR / "TTC Bus Delay Data since 2025.csv")
df_csv = df_csv.drop(columns=["_id"], errors="ignore")
print(f"  2025+ rows : {len(df_csv):,} rows")
print(f"  Columns : {df_csv.columns.tolist()}")
print(df_csv.head(3).to_string())

  2025+ rows : 69,037 rows
  Columns : ['Date', 'Line', 'Time', 'Day', 'Station', 'Code', 'Min Delay', 'Min Gap', 'Bound', 'Vehicle']
                  Date              Line   Time        Day            Station   Code  Min Delay  Min Gap Bound  Vehicle
0  2025-01-01T00:00:00  102 MARKHAM ROAD  02:15  Wednesday     WARDEN STATION  MFESA         20       40     N     3442
1  2025-01-01T00:00:00     65 PARLIAMENT  02:15  Wednesday    KIPLING STATION   MFUS          0        0   NaN        0
2  2025-01-01T00:00:00           64 MAIN  02:40  Wednesday  BROADVIEW STATION   MFUI          0        0   NaN     8546
